# Chương 4: Thực hành - Đại số tuyến tính cho Học máy (Linear Algebra for Machine Learning)

**Mục tiêu bài thực hành:**
1. **Hình học hóa ma trận:** Trực quan hóa các phép biến đổi tuyến tính trong không gian 2D (Scaling, Rotation, Shearing).
2. **Đo lường tương đồng:** Cài đặt các hàm tính khoảng cách chuẩn $L_1$, $L_2$ và độ tương đồng Cosine trên các vector nhúng (Embeddings).
3. **Trị riêng và Vector riêng:** Cài đặt thuật toán Power Iteration tìm vector riêng trội và trực quan hóa các hướng bất biến.
4. **Nén dữ liệu bằng SVD (Singular Value Decomposition):** Triển khai xấp xỉ ma trận hạng thấp ($k$-rank approximation) để nén ảnh và phân tích tỷ lệ bảo toàn năng lượng phổ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Trực quan hóa Phép biến đổi tuyến tính trong Không gian 2D

Một ma trận $\mathbf{A} \in \mathbb{R}^{2 \times 2}$ tác động lên không gian $\mathbb{R}^2$ qua phép nhân vector: $\mathbf{y} = \mathbf{A}\mathbf{x}$.  
Ta vẽ lưới tọa độ ban đầu và theo dõi sự biến dạng sau phép biến đổi.

In [ ]:
# Tạo lưới điểm 2D chuẩn
x_vals = np.linspace(-2, 2, 9)
y_vals = np.linspace(-2, 2, 9)
X, Y = np.meshgrid(x_vals, y_vals)
grid_points = np.vstack([X.ravel(), Y.ravel()]) # 2 x N

# Định nghĩa 3 ma trận biến đổi không gian
# 1. Phép xoay góc 45 độ (Rotation)
theta = np.pi / 4
A_rot = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

# 2. Phép trượt dạt (Shear)
A_shear = np.array([
    [1.0, 1.2],
    [0.0, 1.0]
])

# 3. Phép co giãn không đối xứng (Anisotropic Scale)
A_scale = np.array([
    [1.8, 0.0],
    [0.0, 0.5]
])

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
transforms = [
    ("Không gian ban đầu (I)", np.eye(2)),
    ("Xoay 45 độ (Rotation)", A_rot),
    ("Trượt biến dạng (Shear)", A_shear),
    ("Co giãn (Scaling)", A_scale)
]

for idx, (title, matrix) in enumerate(transforms):
    ax = axes[idx]
    # Áp dụng phép biến đổi: Y = A * X
    trans_grid = matrix @ grid_points
    
    # Vẽ lưới điểm
    ax.scatter(trans_grid[0, :], trans_grid[1, :], color='#3498db', s=30, alpha=0.7)
    
    # Vẽ hai vector cơ sở chuẩn đã biến đổi
    e1 = matrix @ np.array([1, 0])
    e2 = matrix @ np.array([0, 1])
    ax.quiver(0, 0, e1[0], e1[1], angles='xy', scale_units='xy', scale=1, color='red', label='A * e1', width=0.015)
    ax.quiver(0, 0, e2[0], e2[1], angles='xy', scale_units='xy', scale=1, color='green', label='A * e2', width=0.015)
    
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f"{title}\nDet = {np.linalg.det(matrix):.2f}", fontsize=11, fontweight='bold')
    ax.set_aspect('equal')
    if idx == 0:
        ax.legend(loc='upper left', fontsize=9)

plt.suptitle("Hình học hóa Phép biến đổi Ma trận trong Không gian 2D", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Đo lường tương đồng trên Vector đặc trưng (Distance & Cosine Similarity)

Xây dựng hệ thống tìm kiếm tài liệu tương đồng trong không gian nhúng ngữ nghĩa (Semantic Embedding Space).

In [ ]:
def norm_l1(u, v):
    return np.sum(np.abs(u - v))

def norm_l2(u, v):
    return np.sqrt(np.sum((u - v)**2))

def cosine_similarity(u, v):
    dot = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return dot / (norm_u * norm_v)

# Giả lập 4 vector nhúng tài liệu (chiều d=5)
# Doc 1 và Doc 2 cùng chủ đề Học máy; Doc 3 về Ẩm thực; Query là câu hỏi về AI
doc1 = np.array([0.8, 0.9, 0.1, 0.0, 0.2]) # Doc 1: Sách Deep Learning
doc2 = np.array([0.7, 0.8, 0.0, 0.1, 0.3]) # Doc 2: Bài báo Machine Learning
doc3 = np.array([0.0, 0.1, 0.9, 0.8, 0.7]) # Doc 3: Công thức nấu phở
query = np.array([0.85, 0.75, 0.05, 0.0, 0.15]) # Query: 'Thuật toán học sâu'

docs = [("Doc 1 (Deep Learning)", doc1), ("Doc 2 (Machine Learning)", doc2), ("Doc 3 (Ẩm thực Phở)", doc3)]

print("=== ĐO LƯỜNG ĐỘ TƯƠNG ĐỒNG VỚI TRUY VẤN (QUERY) ===")
for name, vec in docs:
    l1_dist = norm_l1(query, vec)
    l2_dist = norm_l2(query, vec)
    cos_sim = cosine_similarity(query, vec)
    print(f"{name:26s} | L1: {l1_dist:.3f} | L2: {l2_dist:.3f} | Cosine Sim: {cos_sim:.4f}")

## 3. Trị riêng và Vector riêng (Power Iteration Algorithm)

Tìm vector riêng tương ứng với trị riêng có độ lớn tuyệt đối lớn nhất mà không cần giải phương trình đặc trưng $\det(\mathbf{A} - \lambda \mathbf{I}) = 0$.

In [ ]:
def power_iteration(A, num_simulations=50):
    # Khởi tạo vector ngẫu nhiên
    b_k = np.random.rand(A.shape[1])
    b_k = b_k / np.linalg.norm(b_k)
    
    for _ in range(num_simulations):
        # Tính tích Ab
        b_k1 = np.dot(A, b_k)
        # Chuẩn hóa độ dài
        b_k = b_k1 / np.linalg.norm(b_k1)
        
    # Ước lượng trị riêng bằng tỉ số Rayleigh: lambda = (b^T A b) / (b^T b)
    eigenvalue = np.dot(b_k.T, np.dot(A, b_k)) / np.dot(b_k.T, b_k)
    return eigenvalue, b_k

# Ma trận hiệp phương sai 2x2 mẫu
A_cov = np.array([
    [3.0, 1.0],
    [1.0, 2.0]
])

eig_val_est, eig_vec_est = power_iteration(A_cov)
true_vals, true_vecs = np.linalg.eigh(A_cov)

print(f"Trị riêng trội ước lượng (Power Iteration): {eig_val_est:.4f}")
print(f"Trị riêng trội chính xác (NumPy eigh)   : {true_vals[-1]:.4f}")
print(f"Vector riêng tương ứng: {eig_vec_est}")

## 4. Phân tích giá trị suy biến (SVD) và Nén ảnh ma trận

Áp dụng định lý Eckart-Young-Mirsky để xấp xỉ ma trận hạng thấp:
$$\mathbf{A}_k = \sum_{i=1}^k \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$
Tỷ lệ năng lượng phổ được bảo toàn:
$$\text{Energy}(k) = \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2} \times 100\%$$

In [ ]:
# Tạo một bức ảnh 2D nhân tạo kích thước 100 x 100 pixel
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
# Bức ảnh dạng hình tròn giao thoa sóng
img = np.sin(np.sqrt(X**2 + Y**2) * 3) + np.cos(X * 2)

# Thực hiện phân tích SVD
U, S, Vt = np.linalg.svd(img, full_matrices=False)

total_energy = np.sum(S**2)
k_ranks = [2, 6, 15, 40]

fig, axes = plt.subplots(1, len(k_ranks) + 1, figsize=(18, 4))

# Ảnh gốc
axes[0].imshow(img, cmap='magma')
axes[0].set_title(f"Ảnh gốc\nHạng = {min(img.shape)}", fontsize=11, fontweight='bold')
axes[0].axis('off')

for idx, k in enumerate(k_ranks, 1):
    # Tái tạo xấp xỉ hạng k
    img_k = np.dot(U[:, :k], np.dot(np.diag(S[:k]), Vt[:k, :]))
    energy_k = (np.sum(S[:k]**2) / total_energy) * 100.0
    # Tính tỷ lệ nén dung lượng: (k*(m + n + 1)) / (m*n)
    compression_ratio = (k * (img.shape[0] + img.shape[1] + 1)) / (img.shape[0] * img.shape[1]) * 100.0
    
    axes[idx].imshow(img_k, cmap='magma')
    axes[idx].set_title(f"Hạng k = {k}\nBảo toàn: {energy_k:.1f}%\nDung lượng: {compression_ratio:.1f}%", fontsize=10)
    axes[idx].axis('off')

plt.suptitle("Nén ảnh bằng Phân tích Giá trị Suy biến SVD (Low-Rank Approximation)", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

## 5. Phân tích Số điều kiện (Condition Number) và Độ ổn định số học

Số điều kiện $\kappa(\mathbf{A}) = \frac{\sigma_{\max}}{\sigma_{\min}}$ phản ánh độ nhạy cảm của nghiệm phương trình $\mathbf{A}\mathbf{x} = \mathbf{b}$ trước các nhiễu nhỏ.

In [ ]:
# Ma trận điều kiện tốt (Well-conditioned)
A_well = np.array([
    [2.0, 0.5],
    [0.5, 2.0]
])

# Ma trận điều kiện xấu (Ill-conditioned) - hai hàng gần như phụ thuộc tuyến tính
A_ill = np.array([
    [1.000, 2.000],
    [1.000, 2.001]
])

cond_well = np.linalg.cond(A_well)
cond_ill = np.linalg.cond(A_ill)

print(f"Số điều kiện Ma trận Tốt (Well-conditioned): kappa = {cond_well:.2f}")
print(f"Số điều kiện Ma trận Xấu (Ill-conditioned) : kappa = {cond_ill:.2f}")

# Thí nghiệm thêm một nhiễu cực nhỏ delta_b = 0.001 vào vector b
b_true = np.array([3.0, 3.001])
b_noisy = b_true + np.array([0.000, 0.001])

x_true = np.linalg.solve(A_ill, b_true)
x_noisy = np.linalg.solve(A_ill, b_noisy)

print("\n--- Tác động của nhiễu trên ma trận điều kiện xấu ---")
print(f"Nghiệm gốc    x_true : {x_true}")
print(f"Nghiệm bị biến động x  : {x_noisy}")
print(f"Sai lệch đầu vào: {np.linalg.norm(b_noisy - b_true):.4f} -> Sai lệch nghiệm: {np.linalg.norm(x_noisy - x_true):.4f}")

## 6. Tổng kết bài học thực hành

1. **Phép nhân ma trận:** Về bản chất là sự biến đổi tọa độ hình học (xoay, kéo giãn, trượt dạt không gian).
2. **Khoảng cách và Tương đồng:** Khoảng cách Euclidean bị chi phối bởi độ dài vector; Cosine Similarity chỉ đo hướng, là chuẩn mực trong không gian nhúng của LLMs và hệ thống tìm kiếm thông tin.
3. **SVD:** Cung cấp công cụ vạn năng để xấp xỉ dữ liệu nhiều chiều thành biểu diễn cô đọng, loại bỏ nhiễu và bảo toàn phần lớn năng lượng thông tin.
4. **Số điều kiện:** Cảnh báo các vấn đề mất ổn định số học trong huấn luyện mô hình sâu, đặt tiền đề cho việc chuẩn hóa dữ liệu (Feature Scaling) và chuẩn hóa tầng (LayerNorm / BatchNorm).